# Pima Indians Diabetes: drzewo decyzyjne i prawidłowa imputacja zer

**Rozszerzenie notebooka `PimaDecisionTree.ipynb`.**  
Cel: pokazać, jak potraktować niewiarygodne fizjologicznie wartości `0` jako braki pomiarów, uniknąć *data leakage* i porównać jakość drzewa decyzyjnego przed oraz po imputacji.

**Plan ćwiczenia:** wczytanie danych → diagnostyka zer → oznaczenie braków → podział 75/25 → imputacja medianą wewnątrz `Pipeline` → trening → macierze pomyłek → porównanie → walidacja krzyżowa → interpretacja drzewa.

**Ważne:** To ćwiczenie edukacyjne, nie narzędzie do diagnozowania pacjentów.

## 1. Import bibliotek

Wykorzystujemy ten sam klasyfikator co w oryginale: `DecisionTreeClassifier(max_depth=4, criterion="gini")`. Dodatkowo przyda się `SimpleImputer` i `Pipeline`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix
)

RANDOM_STATE = 1
pd.set_option("display.max_columns", 20)

## 2. Wczytanie zbioru danych

Notebook wykrywa `diabetes.csv` lub `pima-indians-diabetes.data.csv` w bieżącym katalogu. Obsługuje plik **z nagłówkiem albo bez nagłówka** (tak jak w oryginalnym notatniku). Jeżeli pliku nie ma, spróbuje pobrać publiczną kopię zbioru.

**Źródło kopii danych:** `jbrownlee/Datasets`, plik `pima-indians-diabetes.data.csv` (768 wierszy). Aby pracować offline, umieść ten plik obok notebooka, nadając mu nazwę `diabetes.csv`.

In [ ]:
COL_NAMES = ['pregnant','glucose','bp','skin','insulin','bmi','pedigree','age','label']
DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

candidate_files = [Path("diabetes.csv"), Path("pima-indians-diabetes.data.csv")]
data_path = next((p for p in candidate_files if p.is_file()), None)
source = str(data_path) if data_path else DATA_URL

try:
    raw = pd.read_csv(source, header=None)
except Exception as exc:
    raise FileNotFoundError(
        "Brakuje pliku diabetes.csv w katalogu notebooka. "
        "Pobierz pima-indians-diabetes.data.csv z repozytorium jbrownlee/Datasets "
        "i zapisz go jako diabetes.csv."
    ) from exc

if raw.shape[1] != 9:
    raise ValueError(f"Oczekiwano 9 kolumn, otrzymano {raw.shape[1]}: sprawdź plik CSV.")

# Jeśli pierwszy wiersz zawiera nazwy kolumn, usuwamy go.
first_row_numeric = pd.to_numeric(raw.iloc[0], errors="coerce")
if first_row_numeric.isna().all():
    raw = raw.iloc[1:].reset_index(drop=True)
elif first_row_numeric.isna().any():
    raise ValueError("Nieznany format pierwszego wiersza CSV.")

raw.columns = COL_NAMES
pima = raw.apply(pd.to_numeric, errors="raise").reset_index(drop=True)
pima['label'] = pima['label'].astype(int)
assert set(pima['label'].unique()).issubset({0,1}), 'Etykieta musi być 0 albo 1'

print("Źródło:", source)
print("Wymiary:", pima.shape)
display(pima.head())
print("Rozkład klas:\n", pima['label'].value_counts().sort_index())

## 3. Analiza zer: co jest brakiem, a co poprawną wartością?

W tym zbiorze **zero w `pregnant` jest normalne**. Natomiast `0` w `glucose`, `bp`, `skin`, `insulin` i `bmi` traktujemy jako nieprawidłowo zakodowany brak pomiaru. Nie zamieniamy zer w etykiecie `label`.

Jeżeli w danych brakuje np. prawie połowy pomiarów insuliny, sama metoda uzupełnienia nie naprawi w pełni problemu jakości danych.

In [ ]:
ZERO_AS_MISSING = ['glucose', 'bp', 'skin', 'insulin', 'bmi']

zero_report = pd.DataFrame({
    'zera': (pima[ZERO_AS_MISSING] == 0).sum(),
    'odsetek_%': (pima[ZERO_AS_MISSING] == 0).mean().mul(100).round(2)
})
display(zero_report)
print("Poprawne zera w liczbie ciąż:", int((pima['pregnant'] == 0).sum()))

## 4. Zamiana zer na `NaN` (bez liczenia mediany!)

To **nie jest jeszcze imputacja**. Zamieniamy tylko umowny kod braku danych na `np.nan`. Taka deterministyczna zamiana nie uczy się żadnego parametru z danych, więc może zostać wykonana przed podziałem zbioru.

In [ ]:
pima_missing = pima.copy()
pima_missing[ZERO_AS_MISSING] = pima_missing[ZERO_AS_MISSING].replace(0, np.nan)

print("Liczba braków po zamianie zer na NaN:")
print(pima_missing[ZERO_AS_MISSING].isna().sum())
print("\nBraki w etykiecie:", int(pima_missing['label'].isna().sum()))

## 5. Wybór cech i podział na trening oraz test

W oryginalnym notebooku było **7 cech** (bez `skin`). Zachowujemy je, żeby porównać efekty imputacji z oryginalnym rozwiązaniem. Osobno ocenimy **wszystkie 8 cech**.

Stosujemy ten sam podział `test_size=0.25`, `random_state=1`, co w przesłanym pliku. W tej części nie używamy `stratify`, żeby **nie zmieniać składu zbioru testowego względem oryginału**. W części z walidacją krzyżową zastosujemy podział stratyfikowany.

In [ ]:
ORIGINAL_FEATURES = ['pregnant', 'insulin', 'bmi', 'age', 'glucose', 'bp', 'pedigree']
ALL_FEATURES = ['pregnant', 'glucose', 'bp', 'skin', 'insulin', 'bmi', 'pedigree', 'age']

y = pima['label']
train_idx, test_idx = train_test_split(
    pima.index, test_size=0.25, random_state=RANDOM_STATE
)

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

X_train_raw = pima.loc[train_idx, ORIGINAL_FEATURES]
X_test_raw = pima.loc[test_idx, ORIGINAL_FEATURES]
X_train = pima_missing.loc[train_idx, ORIGINAL_FEATURES]
X_test = pima_missing.loc[test_idx, ORIGINAL_FEATURES]

print("Trening:", len(train_idx), "Test:", len(test_idx))
print("Rozkład klas (trening):\n", y_train.value_counts(normalize=True).round(3))
print("Rozkład klas (test):\n", y_test.value_counts(normalize=True).round(3))

## 6. Dwa modele: oryginalny i z imputacją medianą

### Kluczowa zasada: `Pipeline`

`SimpleImputer(strategy='median')` jest wewnątrz potoku. `fit(X_train, y_train)` liczy mediany **tylko na treningu**, a późniejsze `predict(X_test)` uzupełnia braki w teście **tymi samymi medianami**. To eliminuje wyciek informacji.

Drzewo decyzyjne **nie wymaga standaryzacji** cech, dlatego nie dokładamy tutaj `StandardScaler`.

In [ ]:
# Punkt odniesienia: rozwiązanie z oryginalnego notebooka, bez imputacji
baseline = DecisionTreeClassifier(
    max_depth=4, splitter='best', criterion='gini', random_state=RANDOM_STATE
)
baseline.fit(X_train_raw, y_train)

# Właściwy model: imputacja medianą + identyczne drzewo
tree_imputed = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('tree', DecisionTreeClassifier(
        max_depth=4, splitter='best', criterion='gini', random_state=RANDOM_STATE
    ))
])
tree_imputed.fit(X_train, y_train)

print('Model bez imputacji: gotowy')
print('Model po imputacji: gotowy')

## 7. Jakie wartości model wpisał w miejsce zer?

Ponieważ imputator ma dostęp wyłącznie do obserwacji treningowych, możemy sprawdzić mediany z `named_steps['imputer'].statistics_`.

In [ ]:
imputer = tree_imputed.named_steps['imputer']
medians = pd.Series(imputer.statistics_, index=ORIGINAL_FEATURES, name='mediana_treningowa')
display(medians.to_frame())

# Przykład: kilka oryginalnych rekordów z zerową insuliną (z części treningowej)
rows = X_train_raw.index[X_train_raw['insulin'].eq(0)][:5]
if len(rows):
    before = X_train_raw.loc[rows, ['insulin', 'bmi', 'glucose']].copy()
    after_matrix = imputer.transform(X_train.loc[rows])
    after = pd.DataFrame(after_matrix, index=rows, columns=ORIGINAL_FEATURES)[['insulin', 'bmi', 'glucose']]
    display(pd.concat({'przed': before, 'po_imputacji': after}, axis=1))

## 8. Ocena modeli: więcej niż accuracy

W medycynie **false negative** (osoba z cukrzycą zaklasyfikowana do klasy 0) może być szczególnie istotny. Dlatego pokazujemy macierz pomyłek, `recall`, `precision`, `F1`, `ROC-AUC`, `balanced accuracy`.

**Uwaga:** `ROC-AUC` liczymy z `predict_proba`, a nie z twardych klas `predict`.

In [ ]:
def evaluate_model(name, fitted_model, X_te, y_te):
    predictions = fitted_model.predict(X_te)
    probabilities = fitted_model.predict_proba(X_te)[:, 1]
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_te, predictions),
        'Balanced accuracy': balanced_accuracy_score(y_te, predictions),
        'Precision (1)': precision_score(y_te, predictions, zero_division=0),
        'Recall (1)': recall_score(y_te, predictions, zero_division=0),
        'F1 (1)': f1_score(y_te, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_te, probabilities)
    }

results = [
    evaluate_model('Drzewo oryginalne, 7 cech', baseline, X_test_raw, y_test),
    evaluate_model('Drzewo + imputacja, 7 cech', tree_imputed, X_test, y_test)
]
report = pd.DataFrame(results).set_index('Model')
display(report.round(3))

print('RAPORT KLASYFIKACJI, DRZEWO Z IMPUTACJĄ:\n')
print(classification_report(y_test, tree_imputed.predict(X_test), zero_division=0))

## 9. Macierze pomyłek przed i po imputacji

Konwencja: **wiersze = prawdziwa klasa**, **kolumny = przewidziana klasa**. W klasie 1 najważniejsza jest m.in. liczba pomyłek w komórce `(1, 0)`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, name, model, x in [
    (axes[0], 'Bez imputacji', baseline, X_test_raw),
    (axes[1], 'Po imputacji medianą', tree_imputed, X_test)
]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, model.predict(x), display_labels=['Brak cukrzycy (0)', 'Cukrzyca (1)'],
        ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(name)
fig.suptitle('To samo drzewo i ten sam podział danych')
plt.tight_layout()
plt.show()

## 10. Wariant z pełnym zestawem 8 cech (łącznie z `skin`)

Oryginał pomijał `skin`. Sprawdźmy, czy dodanie tej cechy poprawia wyniki. Skład treningu i testu oraz parametry drzewa pozostają identyczne.

In [ ]:
X_train_all = pima_missing.loc[train_idx, ALL_FEATURES]
X_test_all = pima_missing.loc[test_idx, ALL_FEATURES]

tree_all8 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('tree', DecisionTreeClassifier(max_depth=4, criterion='gini', random_state=RANDOM_STATE))
])
tree_all8.fit(X_train_all, y_train)

results.append(evaluate_model('Drzewo + imputacja, 8 cech', tree_all8, X_test_all, y_test))
report = pd.DataFrame(results).set_index('Model')
display(report.round(3))

## 11. Interpretacja drzewa i ważność cech

Drzewo tworzy sekwencję warunków. Dzięki temu można zobaczyć, **jakie progi i cechy wybrało**. Wagi cech (`feature_importances_`) wskazują spadek nieczystości Gini, a **nie związki przyczynowe**.

In [ ]:
fitted_tree = tree_all8.named_steps['tree']

fig, ax = plt.subplots(figsize=(19, 9))
plot_tree(
    fitted_tree,
    feature_names=ALL_FEATURES,
    class_names=['0', '1'],
    filled=True, rounded=True, fontsize=8, ax=ax
)
ax.set_title('Drzewo decyzyjne po imputacji zer, 8 cech')
plt.tight_layout()
plt.show()

importance = pd.Series(fitted_tree.feature_importances_, index=ALL_FEATURES)
display(importance.sort_values(ascending=False).to_frame('ważność (Gini)'))
print('Reguły drzewa:\n')
print(export_text(fitted_tree, feature_names=ALL_FEATURES, max_depth=4))

## 12. Walidacja krzyżowa, również bez wycieku danych

Pojedynczy podział 75/25 może dać wynik zależny od losowania. `StratifiedKFold` zapewnia podobne proporcje klas w foldach. Cały potok, łącznie z imputacją, jest ponownie dopasowywany **osobno w każdym foldzie**.

**Uwaga techniczna:** konwersja zer → `NaN` jest bezparametrowa. Imputator pozostaje wewnątrz `Pipeline`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
metrics_cv = {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy',
              'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
cv_scores = cross_validate(
    Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('tree', DecisionTreeClassifier(max_depth=4, criterion='gini', random_state=RANDOM_STATE))
    ]),
    pima_missing[ALL_FEATURES], pima_missing['label'],
    scoring=metrics_cv, cv=cv, n_jobs=1
)

cv_table = pd.DataFrame({
    key.replace('test_', ''): [values.mean(), values.std()]
    for key, values in cv_scores.items() if key.startswith('test_')
}, index=['średnia', 'odchylenie std.']).T

display(cv_table.round(3))

## 13. Opcjonalnie: predykcja nowej obserwacji

Wprowadzamy kompletny rekord w **tej samej kolejności kolumn** co w treningu. Jeżeli nie znamy np. stężenia insuliny, podajemy `np.nan`. Potok sam użyje mediany treningowej.

**Wynik modelu nie jest diagnozą kliniczną.**

In [ ]:
nowa_osoba = pd.DataFrame([{
    'pregnant': 2,
    'glucose': 135,
    'bp': 78,
    'skin': np.nan,       # brak pomiaru
    'insulin': np.nan,    # brak pomiaru
    'bmi': 31.2,
    'pedigree': 0.47,
    'age': 44
}], columns=ALL_FEATURES)

prediction = int(tree_all8.predict(nowa_osoba)[0])
probability = float(tree_all8.predict_proba(nowa_osoba)[0, 1])
print('Przewidziana klasa:', prediction)
print('Oszacowane P(klasa=1):', round(probability, 3))
print('Uwaga: prawdopodobieństwa z pojedynczego drzewa nie muszą być dobrze skalibrowane.')

## 14. Pytania dla uczestników i wnioski

1. Dlaczego `pregnant = 0` to poprawna wartość, a `glucose = 0` oznacza brak pomiaru?
2. Co zmieniłoby się, gdyby medianę obliczyć przed podziałem danych?
3. Dlaczego imputator umieszczamy w `Pipeline`, a nie uruchamiamy na całym CSV?
4. Czy po imputacji musi wzrosnąć `accuracy`? **Nie.** Możemy poprawić poprawność metodologiczną, niekoniecznie pojedynczy wynik.
5. Dlaczego `recall` może być ważniejszy niż `accuracy` w analizie przesiewowej?
6. Dlaczego dodanie cechy `skin` nie musi polepszyć wyniku?
7. Dlaczego ważność Gini nie dowodzi, że dana cecha *powoduje* cukrzycę?

**Głębszy wniosek:** algorytm nie rozpoznaje „braku pomiaru” sam z siebie. To projektant systemu określa semantykę danych. Model uczący się na niepoprawnie zakodowanych zerach może być spójny matematycznie, a jednocześnie błędny poznawczo.

**Źródła zbioru:** klasyczny Pima Indians Diabetes Dataset, publiczna kopia: https://github.com/jbrownlee/Datasets/blob/master/pima-indians-diabetes.data.csv